# ECG-image inference in Google Colab

Run the three individual models or their raw-logit ensemble with the paper defaults. This example uses source release **v0.2.1** and the separately published **v0.2.0 weights**. Review the repository's weight terms before downloading; the complete bundle includes the ViT noncommercial condition.

Select a **GPU runtime**. Setup creates a separate inference environment with the tested PyTorch 2.13.0 / torchvision 0.28.0 CUDA 13.0 pair, transformers 5.10.4 and Pillow 13.0.0. Model and image code runs in that environment; the notebook kernel coordinates commands and displays progress.

The default input is a generated **480 x 480 synthetic trace** for checking software execution. To use an existing permitted ECG image, select `INPUT_SOURCE = "file"` and set `USER_IMAGE_PATH` in the input cell. Synthetic predictions have no clinical meaning. Save the prediction and its matching run record using the final download cell.


## 1. Load the source release


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import json
import os
import re
import subprocess
import sys

# A failed source-selection attempt must invalidate earlier success state.
ENVIRONMENT_READY = False
WEIGHTS_READY = False
result = None

if sys.version_info < (3, 11):
    raise RuntimeError("Select a Colab runtime with Python 3.11 or later.")

REPO_URL = "https://github.com/ekagawa007/takotsubo_ami_ai_ecg.git"
GIT_REF = "v0.2.1"  # Available after publication of this release.
REPO_DIR = Path("/content/takotsubo_ami_ai_ecg")
PACKAGE_SOURCE_COMMIT = globals().get("_VERIFIED_PACKAGE_SOURCE_COMMIT")

if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_./-]*", GIT_REF):
    raise ValueError("Set GIT_REF to a branch name, tag name, or commit SHA.")
if REPO_DIR.is_symlink():
    raise RuntimeError("REPO_DIR must not be a symbolic link.")

def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
else:
    if not (REPO_DIR / ".git").is_dir() or (REPO_DIR / ".git").is_symlink():
        raise RuntimeError("REPO_DIR is not a regular Git working directory. Choose another location.")
    if git("remote", "get-url", "origin").rstrip("/") != REPO_URL.rstrip("/"):
        raise RuntimeError("The repository in REPO_DIR does not match REPO_URL.")
    if git("status", "--porcelain"):
        raise RuntimeError("The working directory has changes. Save them before rerunning this cell.")

try:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError("Could not load the requested source revision. Check the log and confirm GIT_REF is published.") from error
CODE_COMMIT = git("rev-parse", "HEAD")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise RuntimeError("Git did not return a complete commit identifier.")

required_names = [
    "requirements-inference.txt", "tools/download_weights.py",
    "src/takotsubo_ami_ai_ecg/checkpoint_compat.py",
    "src/takotsubo_ami_ai_ecg/security.py",
    "src/takotsubo_ami_ai_ecg/image_input.py",
    "src/takotsubo_ami_ai_ecg/safe_output.py",
    "src/takotsubo_ami_ai_ecg/progress.py",
    "src/takotsubo_ami_ai_ecg/assets/weights_v0.2.0.json",
    *[f"examples/predict_{name}.py" for name in ("ensemble", "vit", "convnextv2", "mlpmixer")],
]
required_names = sorted(set(required_names) | {
    p.relative_to(REPO_DIR).as_posix() for p in (REPO_DIR / "src/takotsubo_ami_ai_ecg").glob("*.py")
})
SOURCE_SHA256 = {}
for name in required_names:
    path = REPO_DIR / name
    if any(p.is_symlink() for p in (path, *path.parents)) or not path.is_file():
        raise RuntimeError("The selected source lacks a required regular inference file: " + name)
    SOURCE_SHA256[name] = hashlib.sha256(path.read_bytes()).hexdigest()

def verify_source():
    if REPO_DIR.is_symlink() or not REPO_DIR.is_dir() or git("rev-parse", "HEAD") != CODE_COMMIT:
        raise RuntimeError("The source checkout changed. Start a fresh runtime.")
    if git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("Tracked source files changed. Save your edits and start a fresh runtime.")
    for name, expected in SOURCE_SHA256.items():
        path = REPO_DIR / name
        if any(p.is_symlink() for p in (path, *path.parents)) or not path.is_file():
            raise RuntimeError("A source file is missing or is a symbolic link.")
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise RuntimeError("An inference source file changed. Start a fresh runtime.")

for module_name, module in tuple(sys.modules.items()):
    if module_name == "takotsubo_ami_ai_ecg" or module_name.startswith("takotsubo_ami_ai_ecg."):
        module_file = getattr(module, "__file__", None)
        if (PACKAGE_SOURCE_COMMIT != CODE_COMMIT
                or module_file and not Path(module_file).resolve().is_relative_to(REPO_DIR.resolve())):
            raise RuntimeError("A different package revision is loaded. Start a fresh Colab runtime.")
verify_source()
# Keep this binding unchanged if a later source-selection attempt fails.
_VERIFIED_PACKAGE_SOURCE_COMMIT = CODE_COMMIT
sys.path.insert(0, str(REPO_DIR / "src"))
print("Code commit:", CODE_COMMIT)


## 2. Create the separate inference environment

Select a GPU in the runtime settings. Setup installs the tested CUDA 13.0 pair and inference requirements into `/content/ecg_inference_venv`. It can take several minutes. It does not replace the notebook kernel's ML packages, so a session restart is normally unnecessary.

After a manual restart, run from the first code cell; a completed environment is checked and reused. If installation fails partway, preserve the error log and start a fresh runtime. A partially installed environment is never treated as ready.


In [ ]:
ENVIRONMENT_READY = False
VENV_DIR = Path("/content/ecg_inference_venv")
INFERENCE_PYTHON = VENV_DIR / "bin" / "python"
READY_MARKER = VENV_DIR / "inference_ready.json"
EXPECTED_VERSIONS = {
    "torch": "2.13.0+cu130", "torchvision": "0.28.0+cu130",
    "transformers": "5.10.4", "timm": "1.0.22", "numpy": "2.2.6",
    "Pillow": "13.0.0", "safetensors": "0.7.0", "setuptools": "84.2.1", "pip": "26.2.2",
}
EXPECTED_SETUP = {"code_commit": CODE_COMMIT, "requirements_sha256": SOURCE_SHA256["requirements-inference.txt"], "versions": EXPECTED_VERSIONS}
if VENV_DIR.is_symlink():
    raise RuntimeError("The inference environment path must not be a symbolic link.")
if VENV_DIR.exists():
    if not READY_MARKER.is_file() or READY_MARKER.is_symlink():
        raise RuntimeError("The environment is incomplete. Save the error log and use a fresh Colab runtime.")
    if json.loads(READY_MARKER.read_text(encoding="utf-8")) != EXPECTED_SETUP:
        raise RuntimeError("This environment belongs to a different source revision. Use a fresh runtime.")
else:
    pip_version = tuple(int(v) for v in metadata.version("pip").split(".")[:2])
    if pip_version < (22, 3):
        raise RuntimeError("This setup needs pip 22.3 or later. Select a current Colab runtime.")
    VENV_DIR.mkdir(exist_ok=False)
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(VENV_DIR)], check=True)
    managed_pip = [sys.executable, "-m", "pip", "--python", str(INFERENCE_PYTHON)]
    subprocess.run(managed_pip + ["install", "pip==26.2.2", "setuptools==84.2.1", "--index-url", "https://pypi.org/simple"], check=True)
    subprocess.run(managed_pip + [
        "install", "torch==2.13.0", "torchvision==0.28.0",
        "--index-url", "https://download.pytorch.org/whl/cu130",
    ], check=True)
    subprocess.run(managed_pip + [
        "install", "-r", str(REPO_DIR / "requirements-inference.txt"),
        "--index-url", "https://pypi.org/simple",
    ], check=True)
    subprocess.run([str(INFERENCE_PYTHON), "-m", "pip", "check"], check=True)
    installed = json.loads(subprocess.check_output(
        [str(INFERENCE_PYTHON), "-c",
         "import importlib.metadata as m,json; print(json.dumps({k:m.version(k) for k in " + repr(tuple(EXPECTED_VERSIONS)) + "}))"],
        text=True,
    ))
    if installed != EXPECTED_VERSIONS:
        raise RuntimeError("Inference package versions differ from the reviewed versions.")
    from takotsubo_ami_ai_ecg.safe_output import write_files
    write_files([(READY_MARKER, (json.dumps(EXPECTED_SETUP, indent=2) + "\n").encode("utf-8"))])

subprocess.run([str(INFERENCE_PYTHON), "-m", "pip", "check"], check=True)
ENVIRONMENT_READY = True

def check_inference_environment():
    if not globals().get("ENVIRONMENT_READY", False):
        raise RuntimeError("Run the setup cells in order after opening or restarting the session.")
    if not INFERENCE_PYTHON.is_file() or READY_MARKER.is_symlink():
        raise RuntimeError("The inference environment is unavailable.")
    if json.loads(READY_MARKER.read_text(encoding="utf-8")) != EXPECTED_SETUP:
        raise RuntimeError("The inference environment marker changed.")
    verify_source()

check_inference_environment()
print("Inference environment ready. Notebook-kernel ML packages were not replaced.")
print("Inference interpreter:", INFERENCE_PYTHON)


In [ ]:
check_inference_environment()
runtime_versions = json.loads(subprocess.check_output(
    [str(INFERENCE_PYTHON), "-c", 'import importlib.metadata as metadata\nimport json\nimport sys\nimport torch\nimport torchvision\nfrom PIL import Image\nfrom transformers import ViTConfig, ViTForImageClassification\nversions = {"python": sys.version.split()[0]}\nfor package in ("torch", "torchvision", "timm", "transformers", "numpy", "Pillow", "safetensors", "setuptools", "pip"):\n    versions[package] = metadata.version(package)\nversions["installed_packages"] = sorted([{ "name": d.metadata["Name"], "version": d.version } for d in metadata.distributions() if d.metadata.get("Name")], key=lambda item: item["name"].lower())\nversions["cuda_available"] = torch.cuda.is_available()\nversions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None\nprint(json.dumps(versions))'], text=True,
))
if any(runtime_versions[name] != version for name, version in EXPECTED_VERSIONS.items()):
    raise RuntimeError("The inference interpreter has unexpected package versions.")
if not runtime_versions["cuda_available"]:
    raise RuntimeError("Inference CUDA is unavailable. Select a GPU runtime and inspect the setup log.")
print(json.dumps({k: v for k, v in runtime_versions.items() if k != "installed_packages"}, indent=2))
print("Recorded installed distributions:", len(runtime_versions["installed_packages"]))


## 3. Prepare the weights

The default downloads the public v0.2.0 weights and verifies the archive and every bundled file's SHA-256. The selected checkpoints are unchanged in v0.2.1. To use existing inference weights, set `WEIGHT_SOURCE = "local"` and edit `WEIGHTS_DIR`.

Set `USE_GOOGLE_DRIVE = True` below only if your image or local weights are on Drive. No Drive connection is needed for the synthetic example and automatic download.


In [ ]:
USE_GOOGLE_DRIVE = False  # Set to True only when reading files from Drive.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

In [ ]:
WEIGHTS_READY = False
check_inference_environment()

WEIGHT_SOURCE = "release"  # "release" for verified public weights; "local" for existing files.
WEIGHTS_DIR = Path("/content/ecg_inference_weights")
# Examples for WEIGHT_SOURCE = "local":
# WEIGHTS_DIR = Path("/content/weights")
# WEIGHTS_DIR = Path("/content/drive/MyDrive/ecg_inference/weights")

if WEIGHT_SOURCE not in {"release", "local"}:
    raise ValueError('Set WEIGHT_SOURCE to "release" or "local".')
if WEIGHT_SOURCE == "release":
    download_command = [
        str(INFERENCE_PYTHON), "-u", str(REPO_DIR / "tools" / "download_weights.py"),
        "--output-dir", str(WEIGHTS_DIR),
    ]
    download_process = None
    try:
        download_process = subprocess.Popen(
            download_command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in download_process.stdout:
            print(line, end="", flush=True)
        download_returncode = download_process.wait()
        if download_returncode != 0:
            raise RuntimeError(
                f"Weight download or verification failed with exit code {download_returncode}. "
                "Review the error above. Check the v0.2.0 Release asset or select existing local weights."
            )
    except BaseException:
        if download_process is not None and download_process.poll() is None:
            download_process.terminate()
            try:
                download_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                download_process.kill()
                download_process.wait(timeout=5)
        raise
    finally:
        if download_process is not None and download_process.stdout is not None:
            download_process.stdout.close()
else:
    if not WEIGHTS_DIR.is_dir():
        raise FileNotFoundError("Set WEIGHTS_DIR to the existing inference-weight directory.")
    print("Using local inference weights; no download requested.")

WEIGHTS_READY = True
print("Weights directory:", WEIGHTS_DIR)


## 4. Choose the input image

Keep `INPUT_SOURCE = "synthetic"` for a software check. Select `"file"` and change `USER_IMAGE_PATH` to use an existing permitted image; that file is only read. Enable the Drive cell above when using a Drive path. Changing `USER_IMAGE_PATH` never redirects synthetic generation.

Supported inputs are single-frame PNG, JPEG, BMP, TIFF or WebP. Animated and multipage images are rejected. `strict480` (the default in the next cell) requires prepared **480 x 480 pixel** inputs. Explicit `resize` resizes the entire image bilinearly, without cropping or padding, and can change the aspect ratio.


In [ ]:
check_inference_environment()
INPUT_SOURCE = "synthetic"  # "synthetic" or "file".
USER_IMAGE_PATH = Path("/content/drive/MyDrive/ecg_inference/data/example.png")
SYNTHETIC_IMAGE_PATH = Path("/content/ecg_inference_synthetic.png")

if INPUT_SOURCE == "synthetic":
    IMAGE_PATH = SYNTHETIC_IMAGE_PATH
    input_script = 'from pathlib import Path\nimport hashlib, io, json, math, sys\nsys.path.insert(0, sys.argv[1])\nfrom PIL import Image, ImageDraw\nfrom takotsubo_ami_ai_ecg.security import require_patched_pillow\nfrom takotsubo_ami_ai_ecg.safe_output import write_files\nrequire_patched_pillow()\nimage = Image.new("RGB", (480, 480), "white")\ndraw = ImageDraw.Draw(image)\nfor position in range(0, 480, 12):\n    draw.line((position, 0, position, 479), fill=(255, 225, 225))\n    draw.line((0, position, 479, position), fill=(255, 225, 225))\nfor row in range(4):\n    points = [(x, 60 + row * 110 + int(8 * math.sin(x / 13) - (30 if x % 90 == 45 else 0))) for x in range(480)]\n    draw.line(points, fill=(35, 35, 35), width=2)\npixels_sha = hashlib.sha256(image.tobytes()).hexdigest()\nbuffer = io.BytesIO()\nimage.save(buffer, format="PNG")\nimage_bytes = buffer.getvalue()\npath = Path(sys.argv[2])\nif path.exists() or path.is_symlink():\n    if path.is_symlink() or path.read_bytes() != image_bytes:\n        raise RuntimeError("The synthetic input path contains different data.")\nelse:\n    write_files([(path, image_bytes)])\nprint(json.dumps({"kind": "generated_synthetic_trace", "size": [480,480], "mode": "RGB",\n                  "input_sha256": hashlib.sha256(image_bytes).hexdigest(),\n                  "rgb_pixel_sha256": pixels_sha}))'
elif INPUT_SOURCE == "file":
    IMAGE_PATH = USER_IMAGE_PATH
    if not IMAGE_PATH.is_file():
        raise FileNotFoundError("Set USER_IMAGE_PATH to an existing image; mount Drive above if needed.")
    input_script = 'import hashlib, json, sys\nfrom pathlib import Path\nsys.path.insert(0, sys.argv[1])\nfrom takotsubo_ami_ai_ecg.image_input import open_supported_image\npath = Path(sys.argv[2])\nwith open_supported_image(path, 1) as image:\n    image.load()\n    record = {"kind": "user_provided_image", "size": list(image.size), "mode": image.mode,\n              "rgb_pixel_sha256": hashlib.sha256(image.convert("RGB").tobytes()).hexdigest()}\nrecord["input_sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()\nprint(json.dumps(record))\n'
else:
    raise ValueError('Set INPUT_SOURCE to "synthetic" or "file".')
input_record = json.loads(subprocess.check_output(
    [str(INFERENCE_PYTHON), "-c", input_script, str(REPO_DIR / "src"), str(IMAGE_PATH)], text=True,
))
print("Input:", json.dumps(input_record, indent=2))


## 5. Set inference options

These defaults match the paper. TTA counts include the unchanged view; `1` means no TTA. The ensemble averages raw logits and then applies softmax.


In [ ]:
check_inference_environment()
MODEL = "ensemble"  # "ensemble", "vit", "convnextv2" or "mlpmixer".
OUTPUT_DIR = Path("/content/ecg_inference_results")
VIT_TTA = 24
VIT_MAGNITUDE = 1
CONVNEXTV2_TTA = 1
CONVNEXTV2_MAGNITUDE = 5  # Unused when TTA=1.
MLPMIXER_TTA = 32
MLPMIXER_MAGNITUDE = 3
ENSEMBLE_WEIGHTS = [1.0, 1.0, 1.0]
THRESHOLD = None
SEED = 1234
DEVICE = "cuda"
PRECISION = "float32"
INPUT_POLICY = "strict480"
BATCH_SIZE = 1
NUM_WORKERS = 4


## 6. Check the inputs and run inference

`strict480` requires **480 x 480 pixel** inputs, matching the original dataset evaluation. If your images have a different size, check that the preprocessing matches training. `resize` is an explicit alternative and does not guarantee that the image preparation matches the paper.

The seed controls TTA randomness. However, results may not exactly match the original batch evaluation if image order, batch size, worker count, libraries, or numerical precision differ.
Each run of the inference cell creates a new output filename containing a timestamp and an identifier. Results with no TTA and with the paper's TTA settings can be saved as separate JSON files and run records.

The inference cell displays a live progress bar with overall completion, the current model, TTA view, image count, and elapsed time. The percentage counts completed image/view evaluations; it is not a time estimate and may pause while models load or results are saved. Logs and errors appear as they arrive. Stopping the cell also stops its inference subprocess. The final success message appears after both the prediction and matching run record have been saved.


In [ ]:
check_inference_environment()
if not globals().get("WEIGHTS_READY", False):
    raise RuntimeError("Complete the weight-preparation cell successfully before checking inputs.")

supported_models = ("ensemble", "vit", "convnextv2", "mlpmixer")
if MODEL not in supported_models:
    raise ValueError(f"Choose MODEL from {supported_models}.")
if not IMAGE_PATH.is_file():
    raise FileNotFoundError("Provide the input image and set IMAGE_PATH to its location.")

selected_models = ("vit", "convnextv2", "mlpmixer") if MODEL == "ensemble" else (MODEL,)
required_weights = [WEIGHTS_DIR / model / "model.safetensors" for model in selected_models]
if "vit" in selected_models:
    required_weights.append(WEIGHTS_DIR / "vit" / "config.json")
if any(not path.is_file() for path in required_weights):
    raise FileNotFoundError("Inference weights or configuration files are missing for the selected model(s). Check the layout above.")

input_checked = json.loads(subprocess.check_output(
    [str(INFERENCE_PYTHON), "-c", 'import hashlib, json, sys\nfrom pathlib import Path\nsys.path.insert(0, sys.argv[1])\nfrom takotsubo_ami_ai_ecg.image_input import open_supported_image\npath = Path(sys.argv[2])\nwith open_supported_image(path, 1) as image:\n    size = list(image.size)\n    image.verify()\nprint(json.dumps({"size":size,"input_sha256":hashlib.sha256(path.read_bytes()).hexdigest()}))', str(REPO_DIR / "src"), str(IMAGE_PATH)], text=True,
))
input_size = tuple(input_checked["size"])
if INPUT_POLICY not in {"strict480", "resize"}:
    raise ValueError('Set INPUT_POLICY to "strict480" or "resize".')
if INPUT_POLICY == "strict480" and input_size != (480, 480):
    raise ValueError("strict480 requires 480 x 480 pixels. Prepare the image or explicitly select resize.")
if input_checked["input_sha256"] != input_record["input_sha256"]:
    raise RuntimeError("The input image changed after preparation. Run the input cell again.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input dimensions:", input_size)
print("Selected model:", MODEL)


In [ ]:
result = None

from datetime import datetime, timezone
import hashlib
from html import escape
import math
import time
import uuid
from IPython.display import HTML, display

check_inference_environment()
if not globals().get("WEIGHTS_READY", False):
    raise RuntimeError("Complete the weight-preparation cell successfully before running inference.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_JSON = OUTPUT_DIR / f"prediction_{MODEL}_{stamp}_{uuid.uuid4().hex[:8]}.json"
RUN_JSON = OUTPUT_JSON.with_suffix(".colab_run.json")

command = [
    str(INFERENCE_PYTHON), "-u",
    str(REPO_DIR / "examples" / f"predict_{MODEL}.py"),
    "--image", str(IMAGE_PATH),
    "--weights-dir", str(WEIGHTS_DIR),
    "--output", str(OUTPUT_JSON),
    "--vit-tta", str(VIT_TTA), "--vit-magnitude", str(VIT_MAGNITUDE),
    "--convnextv2-tta", str(CONVNEXTV2_TTA), "--convnextv2-magnitude", str(CONVNEXTV2_MAGNITUDE),
    "--mlpmixer-tta", str(MLPMIXER_TTA), "--mlpmixer-magnitude", str(MLPMIXER_MAGNITUDE),
    "--ensemble-weights", *[str(value) for value in ENSEMBLE_WEIGHTS],
    "--seed", str(SEED),
    "--device", DEVICE,
    "--precision", PRECISION,
    "--input-policy", INPUT_POLICY,
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--progress", "json",
]
if THRESHOLD is not None:
    command.extend(["--threshold", str(THRESHOLD)])

progress_prefix = "ECG_PROGRESS "
progress_state = {"completed": 0, "total": 0, "message": "Starting inference."}
progress_display = None
process = None
progress_error = False
started = time.monotonic()


def finite_number(value, default=0.0):
    try:
        number = float(value)
        return number if math.isfinite(number) else default
    except (TypeError, ValueError):
        return default


def count_pair(current, total):
    if current is None or total is None:
        return "n/a"
    return f"{escape(str(current))}/{escape(str(total))}"


def render_progress(message=None):
    global progress_display
    total = max(0.0, finite_number(progress_state.get("total")))
    completed = max(0.0, finite_number(progress_state.get("completed")))
    percent = min(100.0, 100.0 * completed / total) if total else 0.0
    elapsed = max(time.monotonic() - started, finite_number(progress_state.get("elapsed_seconds")))
    model = progress_state.get("model") or "n/a"
    model_position = count_pair(progress_state.get("model_index"), progress_state.get("model_count"))
    tta = count_pair(progress_state.get("tta_index"), progress_state.get("tta_total"))
    images = count_pair(progress_state.get("images_completed"), progress_state.get("images_total"))
    status = escape(str(message if message is not None else progress_state.get("message", "Running inference.")))
    content = HTML(
        '<div style="padding:10px 0;max-width:760px">'
        f'<div><strong>{status}</strong></div>'
        f'<progress value="{percent:.2f}" max="100" style="width:100%;height:20px" '
        'aria-label="Inference progress"></progress>'
        f'<div>{percent:.1f}% | Model: {escape(str(model))} ({model_position}) '
        f'| TTA: {tta} | Images: {images} | Elapsed: {elapsed:.1f} s</div></div>'
    )
    if progress_display is None:
        progress_display = display(content, display_id=True)
    else:
        progress_display.update(content)


def stop_inference_process():
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)


try:
    render_progress()
    process = subprocess.Popen(
        command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        if line.startswith(progress_prefix):
            try:
                event = json.loads(line[len(progress_prefix):])
            except json.JSONDecodeError:
                print("An unreadable progress update was skipped.", flush=True)
                continue
            if not isinstance(event, dict) or event.get("event") not in {"stage", "progress", "complete", "error"}:
                print("An unrecognized progress update was skipped.", flush=True)
                continue
            progress_state.update(event)
            if event["event"] == "error":
                progress_error = True
                render_progress("Inference failed. Review the logs below.")
            elif event["event"] == "complete":
                render_progress("Prediction saved; finishing the run record.")
            else:
                render_progress()
        else:
            print(line, end="", flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(
            f"Inference stopped with exit code {returncode}. Review the error displayed above."
        )
    if progress_error:
        raise RuntimeError("Inference reported an error. Review the logs above.")

    pending_result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))

    if DEVICE == "cuda" and pending_result["metadata"]["actual_device"] != "cuda":
        raise RuntimeError("The inference did not use CUDA.")
    if any(pending_result["metadata"]["versions"][name] != version
           for name, version in EXPECTED_VERSIONS.items() if name not in {"setuptools", "pip"}):
        raise RuntimeError("Inference did not use the reviewed inference versions.")
    if len(pending_result["predictions"]) != 1 or pending_result["predictions"][0]["input_sha256"] != input_record["input_sha256"]:
        raise RuntimeError("Inference did not use the recorded input image.")
    if hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest() != input_record["input_sha256"]:
        raise RuntimeError("The input changed during inference.")
    # Link the output to its run record without storing an absolute input path.
    run_record = {
        "code_commit": CODE_COMMIT,
        "source_ref": GIT_REF,
        "source_sha256": SOURCE_SHA256,
        "input": input_record,
        "prediction_file": OUTPUT_JSON.name,
        "prediction_sha256": hashlib.sha256(OUTPUT_JSON.read_bytes()).hexdigest(),
        "runtime": runtime_versions,
        "inference_versions": pending_result["metadata"]["versions"],
        "actual_device": pending_result["metadata"]["actual_device"],
    }
    from takotsubo_ami_ai_ecg.safe_output import write_files
    run_bytes = (json.dumps(run_record, ensure_ascii=False, indent=2, allow_nan=False) + "\n").encode("utf-8")
    write_files([(RUN_JSON, run_bytes)], protected_paths=[IMAGE_PATH, OUTPUT_JSON, *required_weights])
    result = pending_result
    render_progress("Inference completed; prediction and run record saved.")
except BaseException as exc:
    result = None
    try:
        stop_inference_process()
    finally:
        failure_message = "Inference cancelled." if isinstance(exc, KeyboardInterrupt) else "Inference failed. Review the error below."
        try:
            render_progress(failure_message)
        except Exception:
            print(failure_message, flush=True)
    raise
finally:
    if process is not None and process.stdout is not None:
        process.stdout.close()

print("Inference completed.")
print("Prediction:", OUTPUT_JSON.name)
print("Run record:", RUN_JSON.name)


The next display shows the saved prediction. The last code cell downloads **one ZIP containing exactly two JSON files**: the prediction and its matching Colab run record. The input image is identified by hashes in the record; images and weights are not included in the result ZIP.

## 7. Review the results

The class order is **0 = tako, 1 = AMI**. By default, the ensemble uses `prob_AMI >= 0.5`, while individual models use argmax (a probability tie is assigned to AMI for the ensemble and tako for an individual model). If you set a numeric `THRESHOLD`, it is used for either case. The following cell displays model outputs. Without comparison against ground-truth labels, these outputs do not establish sensitivity, specificity, or accuracy.

The JSON records the actual execution settings, weight identifiers, logits, and probabilities. Colab's working storage is temporary, so save any results you need to retain.
Successful inference creates `prediction_<model>_<timestamp>_<id>.json` and a matching file ending in `.colab_run.json`. The latter records the code commit, runtime environment, and SHA-256 of the prediction file. Use the download cell below to save both files.


In [ ]:
if not isinstance(globals().get("result"), dict):
    raise RuntimeError("Complete inference successfully before reviewing the results.")

for prediction in result["predictions"]:
    summary = {
        "sample_id": prediction.get("sample_id"),
        "pred_id": prediction["pred_id"],
        "pred_label": prediction["pred_label"],
        "logits": prediction["logits"],
        "probabilities": prediction["probabilities"],
        "models": prediction.get("models", {}),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Download the prediction and its matching run record as one ZIP file.
DOWNLOAD_RESULTS = True
if DOWNLOAD_RESULTS:
    import hashlib
    import json
    from io import BytesIO
    from uuid import uuid4
    from zipfile import ZIP_DEFLATED, ZipFile
    from google.colab import files
    from takotsubo_ami_ai_ecg.safe_output import write_files

    if not isinstance(globals().get("result"), dict) or not OUTPUT_JSON.is_file() or not RUN_JSON.is_file():
        raise RuntimeError("Confirm that inference completed and both JSON files were saved.")

    prediction_bytes = OUTPUT_JSON.read_bytes()
    run_bytes = RUN_JSON.read_bytes()
    record = json.loads(run_bytes)
    if (record.get("prediction_file") != OUTPUT_JSON.name
            or record.get("prediction_sha256") != hashlib.sha256(prediction_bytes).hexdigest()):
        raise RuntimeError("The prediction and run record do not match. Check the selected files.")

    with BytesIO() as buffer:
        with ZipFile(buffer, "w", compression=ZIP_DEFLATED) as archive:
            archive.writestr(OUTPUT_JSON.name, prediction_bytes)
            archive.writestr(RUN_JSON.name, run_bytes)
        zip_bytes = buffer.getvalue()

    ZIP_PATH = OUTPUT_JSON.parent / f"{OUTPUT_JSON.stem}_{uuid4().hex[:8]}.zip"
    write_files([(ZIP_PATH, zip_bytes)], protected_paths=[OUTPUT_JSON, RUN_JSON])
    print("ZIP prepared:", ZIP_PATH.name)
    print("Included:", OUTPUT_JSON.name, "and", RUN_JSON.name)
    files.download(str(ZIP_PATH))


## Validation scope

The tested inference implementation and dependency versions passed a WSL/RTX 5090 float32 no-TTA comparison on ten private ECG images, with no changed classifications and all outputs within the recorded tolerance. A Colab/A100 run using a prepared source archive completed the synthetic-image example with paper TTA, automatic weight download, progress and two linked JSON records on 2026-10-23. The public GitHub source-loading step is checked separately after v0.2.1 publication.

These software checks do not establish clinical performance or numerical equivalence to historical stochastic paper-TTA runs. TTA results depend on image order, batch size, worker count, library versions and precision.

The run record identifies the Git commit, critical source files, input hashes and actual inference environment. It does not contain the absolute input path. Keep records private when their associated images are private, and clear notebook outputs before sharing an executed notebook.
